# Explore TMDb staging output

Checks on the raw TMDb extraction (movies + TV, Indonesian origin):
- how many records came back for each type
- whether any TMDb IDs appear more than once (shouldn't, but worth checking)
- how complete key fields are (release date, overview, poster)
- genre distribution, as a sanity check that this is really Indonesian content

In [ ]:
import json
from pathlib import Path
from collections import Counter

STAGING_DIR = Path("../staging_data/tmdb")

latest_movies = sorted(STAGING_DIR.glob("tmdb_movies_raw_*.json"))[-1]
latest_tv = sorted(STAGING_DIR.glob("tmdb_tv_raw_*.json"))[-1]

print(f"Movies file: {latest_movies.name}")
print(f"TV file:     {latest_tv.name}")

with open(latest_movies, encoding="utf-8") as f:
    movies = json.load(f)

with open(latest_tv, encoding="utf-8") as f:
    tv_shows = json.load(f)

print(f"\nMovies: {len(movies)}")
print(f"TV shows: {len(tv_shows)}")
print(f"Total: {len(movies) + len(tv_shows)}")

In [ ]:
# Check for duplicate TMDb IDs within each set
def check_duplicates(records, label):
    ids = [r["id"] for r in records]
    unique_ids = set(ids)
    print(f"{label}: {len(ids)} total, {len(unique_ids)} unique, {len(ids) - len(unique_ids)} duplicates")
    if len(ids) != len(unique_ids):
        dupes = [tmdb_id for tmdb_id, c in Counter(ids).items() if c > 1]
        print(f"  Duplicate IDs: {dupes[:10]}")

check_duplicates(movies, "Movies")
check_duplicates(tv_shows, "TV shows")

In [ ]:
# Field completeness — how many records are missing key fields
def check_completeness(records, label, fields):
    total = len(records)
    print(f"--- {label} ({total} records) ---")
    for field in fields:
        missing = sum(1 for r in records if not r.get(field))
        present = total - missing
        print(f"  {field:20s} {present}/{total}  ({present/total:.1%} present)")
    print()

check_completeness(movies, "Movies", ["release_date", "overview", "poster_path", "genre_ids", "original_language"])
check_completeness(tv_shows, "TV shows", ["first_air_date", "overview", "poster_path", "genre_ids", "original_language"])

In [ ]:
# Original language breakdown — sanity check that this is really Indonesian content,
# not just things distributed in Indonesia
movie_langs = Counter(r.get("original_language", "?") for r in movies)
tv_langs = Counter(r.get("original_language", "?") for r in tv_shows)

print("Movies by original_language:")
for lang, count in movie_langs.most_common(10):
    print(f"  {lang:5s} {count}")

print("\nTV shows by original_language:")
for lang, count in tv_langs.most_common(10):
    print(f"  {lang:5s} {count}")